In [1]:
import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

%matplotlib inline

In [2]:
df = pd.read_csv('anime.csv')

In [3]:
df.shape

(12294, 7)

In [4]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 12294 entries, 0 to 12293
Data columns (total 7 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   anime_id  12294 non-null  int64  
 1   name      12294 non-null  object 
 2   genre     12232 non-null  object 
 3   type      12269 non-null  object 
 4   episodes  12294 non-null  object 
 5   rating    12064 non-null  float64
 6   members   12294 non-null  int64  
dtypes: float64(1), int64(2), object(4)
memory usage: 672.5+ KB


In [5]:
df.isnull().sum()

anime_id      0
name          0
genre        62
type         25
episodes      0
rating      230
members       0
dtype: int64

In [6]:
df['genre'] = df['genre'].fillna('')
df = df.dropna(subset=['name'])
df = df.reset_index(drop=True)
df.shape

(12294, 7)

In [7]:
tfidf = TfidfVectorizer(stop_words='english')
tfidf_matrix = tfidf.fit_transform(df['genre'])
tfidf_matrix.shape

(12294, 46)

In [8]:
def recommend(title, threshold=0.1, top_n=10):
    match = df[df['name'] == title]
    if match.empty:
        return []
    idx = match.index[0]
    target_vec = tfidf_matrix[idx]
    sim_scores = cosine_similarity(target_vec, tfidf_matrix).flatten()
    sim_indices = np.argsort(sim_scores)[::-1]
    sim_indices = [i for i in sim_indices if i != idx]
    
    recs = []
    for i in sim_indices:
        if sim_scores[i] < threshold:
            break
        recs.append((df.iloc[i]['name'], sim_scores[i]))
        if len(recs) >= top_n:
            break
    return recs

In [9]:
target = df.iloc[0]['name']
print("Recommendations for:", target)
recs = recommend(target, threshold=0.2)
for r in recs:
    print(f"{r[0]} (Score: {r[1]:.4f})")

Recommendations for: Kimi no Na wa.
Wind: A Breath of Heart (TV) (Score: 1.0000)
Wind: A Breath of Heart OVA (Score: 1.0000)
Aura: Maryuuin Kouga Saigo no Tatakai (Score: 0.9553)
Angel Beats!: Another Epilogue (Score: 0.8715)
Harmonie (Score: 0.8715)
Shakugan no Shana (Score: 0.8688)
Shakugan no Shana II (Second) (Score: 0.8688)
Shakugan no Shana S (Score: 0.8688)
Mizuiro (2003) (Score: 0.8548)
Air Movie (Score: 0.8548)


In [10]:
print("Recommendations with low threshold (0.05):")
recs_low = recommend(target, threshold=0.05, top_n=5)
for r in recs_low:
    print(f"{r[0]} (Score: {r[1]:.4f})")

Recommendations with low threshold (0.05):
Wind: A Breath of Heart (TV) (Score: 1.0000)
Wind: A Breath of Heart OVA (Score: 1.0000)
Aura: Maryuuin Kouga Saigo no Tatakai (Score: 0.9553)
Angel Beats!: Another Epilogue (Score: 0.8715)
Harmonie (Score: 0.8715)
